In [1]:
!pip install -U transformers datasets accelerate evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 39.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transf

In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "train.json" in files:
        print("FOUND:", os.path.join(root, "train.json"))

FOUND: /kaggle/input/datasets/ismaeldwikat/reuters/train.json


In [3]:
import sys, os
print(sys.version)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

import transformers
print("transformers:", transformers.__version__)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.9.0+cu126
cuda available: True
transformers: 5.3.0


In [4]:
import json
import pandas as pd

TRAIN_PATH = "/kaggle/input/datasets/ismaeldwikat/reuters/train.json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.head()

shape: (18594, 16)
columns: ['date', 'topics', 'places', 'people', 'organizations', 'exchanges', 'title', 'text', 'keyword', 'domain', 'type', 'quality', 'geographic_information', 'identities', 'title_embedding', 'text_embedding']


,date,topics,places,people,organizations,exchanges,title,text,keyword,domain,type,quality,geographic_information,identities,title_embedding,text_embedding
0,542646302920,[acq],"[usa, beverly-hills, calif.]",[],"[plm-cos-inc, sunlaw-energy-corp-of, january-p...",[],Plm unit ends merger talks,Plm cos inc said its plm power co\nunit broke ...,financial-markets,news,news,medium,"{'city': 'San Francisco', 'state': 'California...",[],"[-0.1740068793, 0.026135793, 0.0245109387, 0.0...","[-0.1289530993, 0.0159782693, 0.0147344209, 0...."
1,542646533080,[coffee],"[london, colombia]",[gilberto-arango],[reuters],[],Colombia opens april/may coffee registrations,Colombia opened coffee export\nregistrations f...,business-news,food-and-drink,news,medium,"{'city': 'Bogotá, Distrito Capital', 'ISO3166-...",[],"[-0.1458295882, -0.0151753156, 0.0214376021, 0...","[-0.0810643211, -0.0274493247, 0.0445648916, 0..."
2,542646712980,"[grain, corn]","[missouri, iowa, minnesota, kansas, north-dako...",[],"[u.s.-agriculture-department, usda]",[],Usda reports 10.572 mln acres in conservation,The u.s. agriculture department has\naccepted ...,business-news,news,news,medium,"{'city': 'Washington', 'state': 'District of C...",[],"[-0.1416630745, 0.0596668795, 0.0581305958, 0....","[-0.1156253815, 0.0484802127, -0.0072592008, -..."
3,542646781160,[],"[usa, brazil, u.s.]","[lawrence-cohn, berger, carole-berger, cohn, r...","[shearson-lehman-brothers-inc, the-securities-...",[],Brazil debt poses thorny issue for u.s. banks,Citicorp appears to be digging\nin its heels ...,industry-news,finance,news,medium,"{'city': 'City of New York', 'state': 'New Yor...","[european, brazilians]","[-0.0938898325, -0.0134521089, 0.0072287978, 0...","[-0.0734064877, -0.0199003667, -0.0317661054, ..."
4,542646790180,[],[usa],[reuter],"[rockwell, rockwell-international-corp]",[],Rockwell to repurchase more common shares,Rockwell international corp said its\nboard ha...,payment-notices,news,news,medium,"{'city': 'Pittsburgh', 'county': 'Allegheny Co...",[],"[-0.1185593829, 0.0217432454, 0.0153466752, 0....","[-0.0806717575, 0.0319725499, -0.0070943153, 0..."


In [5]:
import re

use_cols = ["text", "title", "topics", "quality"]
df2 = df[use_cols].copy()

def clean_text(s: str) -> str:
    s = str(s)
    s = s.replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\sreuter$", "", s, flags=re.IGNORECASE)
    return s

df2["text"]  = df2["text"].apply(clean_text)
df2["title"] = df2["title"].apply(clean_text)

df2 = df2.dropna(subset=["text", "title"])

df2["text_len"]  = df2["text"].str.split().str.len()
df2["title_len"] = df2["title"].str.split().str.len()

df2 = df2[(df2["text_len"] > 30) & (df2["title_len"] > 3)]
df2 = df2[df2["text_len"] < 800]

df2 = df2[df2["quality"].str.lower() != "low"]

print("after cleaning:", df2.shape)
print(df2[["text_len","title_len"]].describe())
df2.head(2)

after cleaning: (12733, 6)
           text_len     title_len
count  12733.000000  12733.000000
mean     143.563339      6.526113
std       90.816414      1.351642
min       31.000000      4.000000
25%       75.000000      6.000000
50%      103.000000      7.000000
75%      195.000000      7.000000
max      364.000000     11.000000


,text,title,topics,quality,text_len,title_len
0,Plm cos inc said its plm power co unit broke o...,Plm unit ends merger talks,[acq],medium,65,5
1,Colombia opened coffee export registrations fo...,Colombia opens april/may coffee registrations,[coffee],medium,65,5


In [6]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df2, test_size=0.1, random_state=42)

print("train:", train_df.shape)
print("val:", val_df.shape)

train: (11459, 6)
val: (1274, 6)


In [7]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[["text","title"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text","title"]], preserve_index=False)

print(train_ds)
print(val_ds)

Dataset({
    features: ['text', 'title'],
    num_rows: 11459
})
Dataset({
    features: ['text', 'title'],
    num_rows: 1274
})


In [8]:
from transformers import BartTokenizer

model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [10]:
max_input_length = 512
max_target_length = 32

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["title"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [11]:
tokenized_train = train_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=train_ds.column_names
)

tokenized_val = val_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=val_ds.column_names
)

print(tokenized_train)
print(tokenized_val)

Map:   0%|          | 0/11459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1274 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 11459
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1274
})


In [12]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [13]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"],
        "rougeLsum": scores["rougeLsum"],
    }

In [14]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/bart-reuters",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

In [15]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer, 
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.207915,0.491512,0.545723,0.296912,0.522071,0.522457
2,0.926958,0.473352,0.561805,0.310503,0.534427,0.534143
3,0.839300,0.465330,0.562816,0.312049,0.535936,0.535764


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1551: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4299, training_loss=1.2520081137856596, metrics={'train_runtime': 2121.7686, 'train_samples_per_second': 16.202, 'train_steps_per_second': 2.026, 'total_flos': 1.048045511245824e+16, 'train_loss': 1.2520081137856596, 'epoch': 3.0})

In [16]:
save_path = "/kaggle/working/bart-reuters-best"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/bart-reuters-best/tokenizer_config.json',
 '/kaggle/working/bart-reuters-best/tokenizer.json')